# AI201 — Programming Assignment 4
## Comparison of Boosted Perceptrons and Support Vector Machines

**Instructor:** Pros Naval  
**Submitted by:** Marcus Rafael B. Tiongson 
**Date:** May 14, 2026

---

## 1. Introduction

This notebook implements and compares two high-performance classification methods: **Boosted Perceptrons** using the AdaBoost algorithm with the Pocket Algorithm as the base learner, and **Support Vector Machines (SVM)** using the scikit-learn library.

AdaBoost is an ensemble method that combines multiple weak learners — each only slightly better than random — into a strong classifier. The Pocket Algorithm is used as the weak learner since the datasets involved are not linearly separable. It extends the standard Perceptron by retaining the best-performing weight vector seen so far during training.

SVMs, on the other hand, are kernel-based methods that implicitly map input data into higher-dimensional feature spaces where an optimal separating hyperplane can be found. This makes them naturally suited for non-linearly separable problems.

The two methods are evaluated on the **Banana** and **Splice** datasets, comparing test accuracy, training speed, and inference speed.

## 2. Imports and Setup

Only `numpy` is used for implementing the Perceptron and AdaBoost algorithms from scratch. `scikit-learn` is used exclusively for the SVM classifier. `matplotlib` handles all plotting.

In [ ]:
# --- Imports ---
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
import sklearn
import time

RANDOM_STATE = 11 #for consistent results
np.random.seed(RANDOM_STATE)
print(f"numpy {np.__version__} | sklearn {sklearn.__version__} | RANDOM_STATE={RANDOM_STATE}")


---
## 3. Part 1: Perceptron Classifier via the Pocket Algorithm

We first build and validate a single Perceptron learner using the Pocket Algorithm before integrating it into the AdaBoost framework. This section uses a simple synthetic two-class dataset to confirm correctness.

### 3.1 Synthetic Dataset Generation

We generate 200 two-dimensional data points drawn from two Gaussian distributions:

- **Class −1:** 100 points from $\mathcal{N}(\mu_1=[0,0]^T,\ \Sigma_1=I)$
- **Class +1:** 100 points from $\mathcal{N}(\mu_2=[10,10]^T,\ \Sigma_2=I)$

Each class is split evenly: 50 points for training and 50 points for testing, yielding a balanced training set of 100 points and a test set of 100 points. Run the code below to generate points.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

X_neg = rng.normal(loc=[0, 0], scale=1.0, size=(100, 2))
X_pos = rng.normal(loc=[10, 10], scale=1.0, size=(100, 2))

X_train_syn = np.vstack([X_neg[:50], X_pos[:50]])
y_train_syn = np.r_[-np.ones(50), np.ones(50)]

X_test_syn = np.vstack([X_neg[50:], X_pos[50:]])
y_test_syn = np.r_[-np.ones(50), np.ones(50)]

plt.figure(figsize=(6, 4))
plt.scatter(X_train_syn[y_train_syn == -1, 0], X_train_syn[y_train_syn == -1, 1], label="Train -1")
plt.scatter(X_train_syn[y_train_syn == 1, 0], X_train_syn[y_train_syn == 1, 1], label="Train +1")
plt.scatter(X_test_syn[y_test_syn == -1, 0], X_test_syn[y_test_syn == -1, 1], marker="x", label="Test -1")
plt.scatter(X_test_syn[y_test_syn == 1, 0], X_test_syn[y_test_syn == 1, 1], marker="x", label="Test +1")
plt.title("Created Gaussian Dataset")
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
plt.savefig("figures/gaussian_dataset.png", bbox_inches="tight", dpi=150)
plt.show()


### 3.2 Pocket Algorithm — `classify()`

The `classify()` function implements the Pocket Algorithm following the assignment pseudocode. At each iteration, a random training sample is selected. If the current vector $\mathbf{v}$ correctly classifies the sample, the consecutive correct-classification counter $n_v$ is incremented. If it misclassifies, then $\mathbf{w}$ is replaced by $\mathbf{v}$ only when $n_v > n_w$. The current vector is then updated using the perceptron rule:

$$v_i \leftarrow v_i + y_j x_{ij}, \quad i = 0, 1, \ldots, d$$

The bias term is implemented by augmenting each sample with $x_0 = 1$. Training runs for `maxitercnt = 10000` iterations, as required.


In [3]:
# in: X (training data), y (labels), maxitercnt
# out: w (best weight vector including bias)
def _add_bias(X):
    return np.c_[np.ones(X.shape[0]), X]

def _sign(z):
    return np.where(z >= 0, 1.0, -1.0)

def accuracy(y_true, y_pred):
    return float(np.mean(y_true == y_pred))

def classify(X, y, maxitercnt=10000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    Xb = _add_bias(X)
    n_samples, n_features = Xb.shape

    v = np.zeros(n_features)
    w = np.zeros(n_features)
    nv = 0
    nw = 0

    random_indices = rng.integers(n_samples, size=maxitercnt + 1) #max 10000

    for j in random_indices:
        yhat = 1.0 if np.dot(v, Xb[j]) >= 0 else -1.0

        if yhat * y[j] > 0:
            nv += 1
        else:
            if nv > nw:
                w = v.copy()
                nw = nv
            v += y[j] * Xb[j]
            nv = 0

    if nv > nw: #classify, put in pocket
        w = v.copy()

    return w


### 3.3 Prediction — `predict()`

The `predict()` function applies the learned weight vector $\mathbf{w}$ to classify unseen data. A sample $\mathbf{x}$ is assigned:

$$\hat{y} = \text{sgn}(\mathbf{w} \cdot \mathbf{x})$$

The function also computes the **sum of squared errors (SSE)** over the test set as a scalar summary of performance.

In [4]:
# in: X (test data), y (true labels), w (weight vector)
# out: predictions, SSE
def predict(X, y=None, w=None):
    predictions = _sign(_add_bias(X) @ w)

    if y is None:
        return predictions

    sse = np.sum((y - predictions) ** 2)
    return predictions, float(sse)


### 3.4 Single Perceptron Evaluation

We train the Pocket Algorithm on the synthetic dataset and evaluate it on the held-out test set.

In [5]:
w_syn = classify(X_train_syn, y_train_syn, maxitercnt=10000, seed=1)

train_pred_syn, train_sse_syn = predict(X_train_syn, y_train_syn, w_syn)
test_pred_syn, test_sse_syn = predict(X_test_syn, y_test_syn, w_syn)

train_acc_syn = accuracy(y_train_syn, train_pred_syn)
test_acc_syn = accuracy(y_test_syn, test_pred_syn)

print(f"Learned weight vector: {w_syn}")
print(f"Training accuracy: {train_acc_syn:.4f}")
print(f"Test accuracy: {test_acc_syn:.4f}")
print(f"Test SSE: {test_sse_syn:.1f}")


Learned weight vector: [-11.           0.6586445    3.84756647]
Training accuracy: 1.0000
Test accuracy: 1.0000
Test SSE: 0.0


### 3.5 Results: Single Perceptron

The Pocket Algorithm was run for up to 10,000 iterations. On the synthetic dataset, the single perceptron achieved:

| Metric | Value |
|---|---:|
| Training Accuracy | 100.00% |
| Test Accuracy | 100.00% |
| Sum of Squared Errors (SSE) | 0.0 |

The high separability of the two Gaussians allowed the Pocket Algorithm to find a nearly perfect decision boundary. This validates the implementation before proceeding to AdaBoost.

---
## 4. Part 2: AdaBoost with Boosted Perceptrons

With the Pocket Algorithm confirmed working, we now embed it as the weak learner inside the AdaBoost framework. The ensemble produces a weighted vote of $K$ perceptrons, where each learner is trained on a reweighted version of the training set that penalizes previously well-classified examples and up-weights misclassified ones.

### 4.1 AdaBoost Training — `adabtrain()`

The `adabtrain()` function implements the full AdaBoost loop over $K$ iterations:

1. Initialize uniform sample weights $w_1(i) = 1/N$
2. At each round $t$: sample $S_t$ from $S$ with replacement using $w_t$, train weak learner $h_t$ via `classify()` using the required 10,000 Pocket iterations, compute weighted error $\epsilon_t$, compute coefficient $\alpha_t = \frac{1}{2}\ln\left(\frac{1 - \epsilon_t}{\epsilon_t}\right)$, and update weights

3. Return the list of hypotheses $h_1, \ldots, h_K$ and their coefficients $alpha_1, \ldots, alpha_K$

Special handling is applied when $\epsilon_t = 0$ or $\epsilon_t \geq 0.5$ to avoid numerical instability and to maintain the weak-learner requirement.


In [6]:
# in: X (training data), y (labels), K (number of learners)
# out: list of (alpha, w) pairs for K hypotheses
def adabtrain(X, y, K=1000, weak_maxiter=10000, seed=RANDOM_STATE, checkpoints=None):
    rng = np.random.default_rng(seed)
    n_samples = X.shape[0]
    weights = np.ones(n_samples) / n_samples
    ensemble = []
    history = {}

    if checkpoints is None:
        checkpoints = set()
    else:
        checkpoints = set(checkpoints)

    for t in range(1, K + 1):
        sample_idx = rng.choice(n_samples, size=n_samples, replace=True, p=weights)
        w_t = classify(
            X[sample_idx],
            y[sample_idx],
            maxitercnt=weak_maxiter,
            seed=int(rng.integers(1_000_000_000)),
        )

        pred = predict(X, w=w_t)
        err = float(np.sum(weights[pred != y]))

        if err <= 1e-12:
            alpha_t = 0.5 * np.log((1 - 1e-12) / 1e-12)
            ensemble.append((float(alpha_t), w_t))
            history[t] = list(ensemble)
            break

        if err >= 0.5:
            flipped_pred = -pred
            flipped_err = float(np.sum(weights[flipped_pred != y]))
            if flipped_err < 0.5:
                w_t = -w_t
                pred = flipped_pred
                err = flipped_err
            else:
                if t in checkpoints:
                    history[t] = list(ensemble)
                continue

        alpha_t = 0.5 * np.log((1 - err) / err)
        weights *= np.exp(-alpha_t * y * pred)
        weights /= weights.sum()

        ensemble.append((float(alpha_t), w_t))

        if t in checkpoints:
            history[t] = list(ensemble)

    return ensemble, history


### 4.2 AdaBoost Prediction — `adabpredict()`

The `adabpredict()` function computes the ensemble output as the sign of the weighted sum of all weak hypotheses:

$$H(\mathbf{x}) = \text{sgn}\left(\sum_{t=1}^{K} \alpha_t h_t(\mathbf{x})\right)$$

Each hypothesis $h_t$ is applied via the learned weight vector $\mathbf{w}_t$ from the Pocket Algorithm.

In [7]:
# in: X (data), ensemble [(alpha_t, w_t), ...]
# out: predicted labels
def adabpredict(X, ensemble):
    if len(ensemble) == 0:
        return np.ones(X.shape[0])

    Xb = _add_bias(X)
    scores = np.zeros(X.shape[0])

    for alpha_t, w_t in ensemble:
        scores += alpha_t * _sign(Xb @ w_t)

    return _sign(scores)


### 4.3 Dataset Loading

We load the **Banana** and **Splice** datasets and prepare the train/test splits as specified:

| Dataset | Training Set | Test Set |
|---|---:|---:|
| Banana | 400 points | 4,900 points |
| Splice | 1,000 points | 2,175 requested; all remaining rows used if the uploaded CSV has fewer than 3,175 rows |

The uploaded Splice CSV contains fewer than 3,175 total rows, so after taking 1,000 training examples the notebook uses all remaining examples as the test set and prints this limitation explicitly.


In [8]:
#load datasets
# CSV format: first column = label {-1, +1}, remaining columns = features.
banana_data = np.loadtxt("data/banana_data.csv", delimiter=",")
splice_data = np.loadtxt("data/splice_data.csv", delimiter=",")

X_banana, y_banana = banana_data[:, 1:], banana_data[:, 0]
X_splice, y_splice = splice_data[:, 1:], splice_data[:, 0]

def stratified_train_test_split_numpy(X, y, train_size, test_size=None, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    classes, counts = np.unique(y, return_counts=True)
    n_total = len(y)
    if test_size is None:
        test_size = n_total - train_size
    feasible_test_size = min(test_size, n_total - train_size)

    train_indices = []
    test_indices = []
    for cls, count in zip(classes, counts):
        cls_idx = np.where(y == cls)[0]
        rng.shuffle(cls_idx)
        cls_train = int(round(train_size * count / n_total))
        cls_train = min(cls_train, len(cls_idx))
        train_indices.extend(cls_idx[:cls_train])
        test_indices.extend(cls_idx[cls_train:])

    # Adjust exact train size if rounding caused a small mismatch.
    train_indices = list(train_indices)
    test_indices = list(test_indices)
    rng.shuffle(train_indices)
    rng.shuffle(test_indices)
    while len(train_indices) < train_size and test_indices:
        train_indices.append(test_indices.pop())
    while len(train_indices) > train_size:
        test_indices.append(train_indices.pop())

    test_indices = test_indices[:feasible_test_size]
    rng.shuffle(train_indices)
    rng.shuffle(test_indices)

    return (
        X[np.array(train_indices)], X[np.array(test_indices)],
        y[np.array(train_indices)], y[np.array(test_indices)],
        feasible_test_size
    )

def standardize_train_test_numpy(X_train_raw, X_test_raw):
    mean = X_train_raw.mean(axis=0)
    std = X_train_raw.std(axis=0)
    std[std == 0] = 1.0
    return (X_train_raw - mean) / std, (X_test_raw - mean) / std

X_banana_train_raw, X_banana_test_raw, y_banana_train, y_banana_test, banana_test_used = stratified_train_test_split_numpy(
    X_banana, y_banana, train_size=400, test_size=4900, seed=RANDOM_STATE
)
X_splice_train_raw, X_splice_test_raw, y_splice_train, y_splice_test, splice_test_used = stratified_train_test_split_numpy(
    X_splice, y_splice, train_size=1000, test_size=2175, seed=RANDOM_STATE
)

X_banana_train, X_banana_test = standardize_train_test_numpy(X_banana_train_raw, X_banana_test_raw)
X_splice_train, X_splice_test = standardize_train_test_numpy(X_splice_train_raw, X_splice_test_raw)

print("Banana train/test:", X_banana_train.shape, X_banana_test.shape)
print("Splice train/test:", X_splice_train.shape, X_splice_test.shape)
if splice_test_used < 2175:
    print(
        f"Splice note: requested 2,175 test points, but splice_data.csv has "
        f"{len(splice_data)} rows total; after 1,000 training points, only "
        f"{splice_test_used} test points are available."
    )


Banana train/test: (400, 2) (4900, 2)
Splice train/test: (1000, 60) (1991, 60)
Splice note: requested 2,175 test points, but splice_data.csv has 2991 rows total; after 1,000 training points, only 1991 test points are available.


### 4.4 Banana Dataset — AdaBoost Evaluation

We run AdaBoost with $K = 10, 20, 30, \ldots, 1000$ learners on the Banana dataset and record training and test accuracies at each value of $K$.

In [ ]:
#adaboost - banana
K_values = np.arange(10, 1001, 10)
checkpoints = set(K_values)

def evaluate_adaboost_dataset(X_train, y_train, X_test, y_test, weak_maxiter=10000):
    start_train = time.perf_counter()
    ensemble, history = adabtrain(
        X_train, y_train, K=1000, weak_maxiter=weak_maxiter,
        seed=123, checkpoints=checkpoints
    )
    train_time = time.perf_counter() - start_train

    for K in K_values:
        history.setdefault(K, ensemble)

    Xb_train = _add_bias(X_train)
    Xb_test = _add_bias(X_test)
    train_scores = np.zeros(X_train.shape[0])
    test_scores = np.zeros(X_test.shape[0])
    train_acc = []
    test_acc = []
    previous_len = 0

    for K in K_values:
        current_ensemble = history[K]
        for alpha_t, w_t in current_ensemble[previous_len:]:
            train_scores += alpha_t * _sign(Xb_train @ w_t)
            test_scores += alpha_t * _sign(Xb_test @ w_t)
        previous_len = len(current_ensemble)
        train_acc.append(accuracy(y_train, _sign(train_scores)))
        test_acc.append(accuracy(y_test, _sign(test_scores)))

    start_infer = time.perf_counter()
    final_test_pred = adabpredict(X_test, ensemble)
    infer_time = time.perf_counter() - start_infer

    return {
        "ensemble": ensemble,
        "train_accuracy": np.array(train_acc),
        "test_accuracy": np.array(test_acc),
        "train_time": train_time,
        "infer_time": infer_time,
        "final_train_accuracy": accuracy(y_train, adabpredict(X_train, ensemble)),
        "final_test_accuracy": accuracy(y_test, final_test_pred),
    }

banana_adaboost = evaluate_adaboost_dataset(
    X_banana_train, y_banana_train, X_banana_test, y_banana_test
)

for K in K_values:
    idx = K // 10 - 1
    print(
        f"K={K:4d} | train={banana_adaboost['train_accuracy'][idx]:.4f} "
        f"| test={banana_adaboost['test_accuracy'][idx]:.4f}"
    )

print(f"Training time: {banana_adaboost['train_time']:.4f} s")
print(f"Inference time: {banana_adaboost['infer_time']:.4f} s")


### 4.5 Banana Dataset — Accuracy vs K Plot

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(K_values, banana_adaboost["train_accuracy"] * 100, label="Training accuracy")
plt.plot(K_values, banana_adaboost["test_accuracy"] * 100, label="Test accuracy")
plt.xlabel("K learners")
plt.ylabel("Accuracy (%)")
plt.title("Banana: Boosted Perceptron Accuracy vs K")
plt.grid(alpha=0.3)
plt.legend()
plt.savefig("figures/banana_adaboost_accuracy.png", bbox_inches="tight", dpi=150)
plt.show()


### 4.6 Results: Boosted Perceptron on Banana Dataset

The plot above shows how training and test accuracies evolve as more weak learners are added to the ensemble. The table below shows every 100th checkpoint plus K=10. See **Appendix A** for the full K=10, 20, …, 1000 table.

| K (Learners) | Train Accuracy | Test Accuracy |
|---:|---:|---:|
| 10 | 74.25% | 76.24% |
| 100 | 87.25% | 86.41% |
| 200 | 89.25% | 88.33% |
| 300 | 89.50% | 88.61% |
| 400 | 90.50% | 89.12% |
| 500 | 90.75% | 89.18% |
| 600 | 91.00% | 89.24% |
| 700 | 91.75% | 89.45% |
| 800 | 91.75% | 89.47% |
| 900 | 92.50% | 89.53% |
| 1000 | 92.75% | 89.18% |

**Observations:**
- Training accuracy increased as more weak learners were added, reaching 92.75% at $K = 1000$.
- Test accuracy improved rapidly in the early ensemble sizes and reached 89.18% at $K = 1000$.
- The test curve plateaued after the early gains, showing diminishing returns from adding many more perceptrons.
- The Banana dataset's curved decision boundary is not ideal for a single linear classifier, but the boosted ensemble captured it reasonably well.

### 4.7 Splice Dataset — AdaBoost Evaluation

We repeat the AdaBoost evaluation on the Splice dataset using the same range of $K$ values.

In [ ]:
# adaboost - splice
splice_adaboost = evaluate_adaboost_dataset(
    X_splice_train, y_splice_train, X_splice_test, y_splice_test
)

for K in K_values:
    idx = K // 10 - 1
    print(
        f"K={K:4d} | train={splice_adaboost['train_accuracy'][idx]:.4f} "
        f"| test={splice_adaboost['test_accuracy'][idx]:.4f}"
    )

print(f"Training time: {splice_adaboost['train_time']:.4f} s")
print(f"Inference time: {splice_adaboost['infer_time']:.4f} s")


### 4.8 Splice Dataset — Accuracy vs K Plot

In [ ]:
#plotting
plt.figure(figsize=(7, 4))
plt.plot(K_values, splice_adaboost["train_accuracy"] * 100, label="Training accuracy")
plt.plot(K_values, splice_adaboost["test_accuracy"] * 100, label="Test accuracy")
plt.xlabel("K learners")
plt.ylabel("Accuracy (%)")
plt.title("Splice: Boosted Perceptron Accuracy vs K")
plt.ylim(0, 100)
plt.grid(alpha=0.3)
plt.legend()
plt.savefig("figures/splice_adaboost_accuracy.png", bbox_inches="tight", dpi=150)
plt.show()


### 4.9 Results: Boosted Perceptron on Splice Dataset

The table below shows every 100th checkpoint plus K=10. See **Appendix A** for the full K=10, 20, …, 1000 table.

| K (Learners) | Train Accuracy | Test Accuracy |
|---:|---:|---:|
| 10 | 84.50% | 80.26% |
| 100 | 86.40% | 80.86% |
| 200 | 88.80% | 81.12% |
| 300 | 93.10% | 81.62% |
| 400 | 95.90% | 82.57% |
| 500 | 98.30% | 82.82% |
| 600 | 99.30% | 83.32% |
| 700 | 99.80% | 83.02% |
| 800 | 100.00% | 83.07% |
| 900 | 100.00% | 83.38% |
| 1000 | 100.00% | 83.17% |

**Observations:**
- The Splice dataset was more challenging because of its higher-dimensional feature space.
- Training accuracy reached 100.00%, while test accuracy reached 83.17% at $K = 1000$.
- The widening gap between training and test accuracy at large $K$ indicates overfitting sensitivity on this dataset.
- The uploaded Splice CSV and LIBSVM files each contain 2,991 rows, so the 1,000-row training split leaves 1,991 test examples rather than the requested 2,175.

---
## 5. Part 3: Support Vector Machine Classifier

We now train an SVM on the same datasets using `sklearn.svm.SVC`. The goal is to find the kernel and hyperparameters that yield the best test accuracy, and then compare the SVM's performance against the Boosted Perceptron.

### 5.1 Kernel Selection and Hyperparameter Search

We evaluate the following kernels available in `sklearn.svm.SVC`:

| Kernel | Key Parameters |
|---|---|
| Linear | `C` |
| RBF (Gaussian) | `C`, `gamma` |
| Polynomial | `C`, `degree`, `coef0` |
| Sigmoid | `C`, `gamma`, `coef0` |

A grid search or manual sweep is performed over the parameter space to identify the best-performing configuration for each dataset.

In [13]:
# SVM
def svm_parameter_search(X_train, y_train, X_test, y_test):
    configs = []

    for C in [0.1, 1, 10, 100]:
        configs.append({"kernel": "linear", "C": C})

    for C in [0.1, 1, 10, 100]:
        for gamma in ["scale", 0.01, 0.1, 1]:
            configs.append({"kernel": "rbf", "C": C, "gamma": gamma})

    for C in [0.1, 1, 10]:
        for degree in [2, 3, 4]:
            configs.append({"kernel": "poly", "C": C, "degree": degree, "gamma": "scale", "coef0": 1})

    for C in [0.1, 1, 10]:
        for gamma in ["scale", 0.01, 0.1]:
            configs.append({"kernel": "sigmoid", "C": C, "gamma": gamma, "coef0": 0})

    rows = []
    best = None

    for config in configs:
        model = SVC(**config)

        start_train = time.perf_counter()
        model.fit(X_train, y_train)
        train_time = time.perf_counter() - start_train

        start_infer = time.perf_counter()
        test_pred = model.predict(X_test)
        infer_time = time.perf_counter() - start_infer

        row = {
            **config,
            "train_accuracy": accuracy(y_train, model.predict(X_train)),
            "test_accuracy": accuracy(y_test, test_pred),
            "train_time": train_time,
            "infer_time": infer_time,
            "model": model,
        }
        rows.append(row)

        if best is None or row["test_accuracy"] > best["test_accuracy"]:
            best = row

    return rows, best

banana_svm_rows, banana_svm_best = svm_parameter_search(
    X_banana_train, y_banana_train, X_banana_test, y_banana_test
)
splice_svm_rows, splice_svm_best = svm_parameter_search(
    X_splice_train, y_splice_train, X_splice_test, y_splice_test
)

def print_best_svm(name, best):
    readable = {k: v for k, v in best.items() if k != "model"}
    print(name, readable)

print_best_svm("Banana best SVM:", banana_svm_best)
print_best_svm("Splice best SVM:", splice_svm_best)


Banana best SVM: {'kernel': 'rbf', 'C': 10, 'gamma': 'scale', 'train_accuracy': 0.9, 'test_accuracy': 0.9014285714285715, 'train_time': 0.003622199999881559, 'infer_time': 0.046641699998872355}
Splice best SVM: {'kernel': 'rbf', 'C': 100, 'gamma': 0.01, 'train_accuracy': 1.0, 'test_accuracy': 0.891511803114013, 'train_time': 0.06015000000115833, 'infer_time': 0.21230459999969753}


### 5.2 SVM on Banana Dataset — Best Config

After sweeping the parameter space, the best-performing SVM configuration for the Banana dataset was identified as:

| Parameter | Value |
|---|---:|
| Kernel | `rbf` |
| C | 10 |
| gamma | `scale` |

#### Per-Kernel Comparison (Banana)

| Kernel | Best Config | Train Accuracy | Test Accuracy |
|---|---|---:|---:|
| Linear | C=0.1 | 55.25% | 55.16% |
| **RBF** | **C=10, gamma=scale** | **90.00%** | **90.14%** |
| Polynomial | C=10, degree=4 | 89.50% | 89.10% |
| Sigmoid | C=0.1, gamma=0.01 | 55.25% | 55.16% |

The Banana dataset has a curved, crescent-shaped decision boundary that no linear transform can separate. This explains why the **linear** and **sigmoid** kernels both stall at ~55% — barely above the 50% random baseline for a balanced binary problem — regardless of how `C` is tuned. They impose a hyperplane in the original (or a sigmoid-warped) feature space, which cannot trace a curved region.

The **polynomial** kernel (degree 4) reaches 89.10% and comes close to RBF, since a high-degree polynomial can approximate curved boundaries. However, polynomial kernels are sensitive to degree choice and can produce unstable gradients for points far from the origin; they also require explicit degree tuning. The **RBF** kernel, by contrast, corresponds to an infinite-dimensional feature space where any smooth decision boundary can be represented. It outperforms the best polynomial config by ~1 percentage point while requiring fewer hyperparameter decisions (only `C` and `gamma`).

Source: https://www.geeksforgeeks.org/machine-learning/radial-basis-function-kernel-machine-learning/

In [14]:
# banana SVM
banana_best_config = {k: v for k, v in banana_svm_best.items() if k not in {
    "model", "train_accuracy", "test_accuracy", "train_time", "infer_time"
}}

start_train = time.perf_counter()
banana_svm = SVC(**banana_best_config)
banana_svm.fit(X_banana_train, y_banana_train)
banana_svm_train_time = time.perf_counter() - start_train

start_infer = time.perf_counter()
banana_svm_test_pred = banana_svm.predict(X_banana_test)
banana_svm_infer_time = time.perf_counter() - start_infer

banana_svm_train_acc = accuracy(y_banana_train, banana_svm.predict(X_banana_train))
banana_svm_test_acc = accuracy(y_banana_test, banana_svm_test_pred)

print("Best Banana SVM config:", banana_best_config)
print(f"Train accuracy: {banana_svm_train_acc:.4f}")
print(f"Test accuracy: {banana_svm_test_acc:.4f}")
print(f"Training time: {banana_svm_train_time:.4f} s")
print(f"Inference time: {banana_svm_infer_time:.4f} s")


Best Banana SVM config: {'kernel': 'rbf', 'C': 10, 'gamma': 'scale'}
Train accuracy: 0.9000
Test accuracy: 0.9014
Training time: 0.0041 s
Inference time: 0.0750 s


#### Results: SVM on Banana Dataset

| Metric | Value |
|---|---:|
| Train Accuracy | 90.00% |
| Test Accuracy | 90.14% |
| Training Time | 0.0056 s |
| Inference Time | 0.0917 s |

### 5.3 SVM on Splice Dataset — Best Config

For the Splice dataset, the best-performing SVM configuration was:

| Parameter | Value |
|---|---:|
| Kernel | `rbf` |
| C | 100 |
| gamma | `0.01` |

#### Per-Kernel Comparison (Splice)

| Kernel | Best Config | Train Accuracy | Test Accuracy |
|---|---|---:|---:|
| Linear | C=1 | 87.10% | 83.83% |
| **RBF** | **C=100, gamma=0.01** | **100.00%** | **89.15%** |
| Polynomial | C=1, degree=3 | 100.00% | 88.30% |
| Sigmoid | C=1, gamma=0.01 | 84.40% | 84.53% |

Unlike Banana, all four kernels achieve reasonable accuracy on Splice because the dataset's 60-dimensional feature space provides richer structure that even a linear hyperplane can exploit partially (83.83%). However, the non-linear interactions between splice-site features mean that kernel methods with higher representational capacity still win.

The **sigmoid** kernel performs similarly to linear (~84%), which is consistent with the known result that the sigmoid kernel does not satisfy Mercer's condition globally and can behave as a pseudo-linear kernel in practice.

The **polynomial** kernel (degree 3) reaches 88.30% and again comes close to RBF, but the RBF kernel's advantage in this higher-dimensional setting is that a small `gamma = 0.01` (wide Gaussian) acts as a soft global smoother over the feature space, effectively capturing long-range feature correlations without overfitting to a specific polynomial degree. RBF leads polynomial by ~0.85 percentage points at test time.

The RBF kernel is therefore the preferred choice across both datasets: it degrades gracefully when the problem is nearly linear (Splice, where all kernels work), and it is the only kernel that avoids catastrophic failure when the boundary is highly curved (Banana, where linear and sigmoid collapse to ~55%).

Source: https://www.geeksforgeeks.org/machine-learning/radial-basis-function-kernel-machine-learning/

In [15]:
# splice SVM
splice_best_config = {k: v for k, v in splice_svm_best.items() if k not in {
    "model", "train_accuracy", "test_accuracy", "train_time", "infer_time"
}}

start_train = time.perf_counter()
splice_svm = SVC(**splice_best_config)
splice_svm.fit(X_splice_train, y_splice_train)
splice_svm_train_time = time.perf_counter() - start_train

start_infer = time.perf_counter()
splice_svm_test_pred = splice_svm.predict(X_splice_test)
splice_svm_infer_time = time.perf_counter() - start_infer

splice_svm_train_acc = accuracy(y_splice_train, splice_svm.predict(X_splice_train))
splice_svm_test_acc = accuracy(y_splice_test, splice_svm_test_pred)

print("Best Splice SVM config:", splice_best_config)
print(f"Train accuracy: {splice_svm_train_acc:.4f}")
print(f"Test accuracy: {splice_svm_test_acc:.4f}")
print(f"Training time: {splice_svm_train_time:.4f} s")
print(f"Inference time: {splice_svm_infer_time:.4f} s")


Best Splice SVM config: {'kernel': 'rbf', 'C': 100, 'gamma': 0.01}
Train accuracy: 1.0000
Test accuracy: 0.8915
Training time: 0.0606 s
Inference time: 0.1825 s


#### Results: SVM on Splice Dataset

| Metric | Value |
|---|---:|
| Train Accuracy | 100.00% |
| Test Accuracy | 89.15% |
| Training Time | 0.0692 s |
| Inference Time | 0.2686 s |

---
## 6. Part 4: Boosted Perceptrons vs SVM

We now directly compare the two methods across three dimensions: **test accuracy**, **training speed**, and **inference speed**.

### 6.1 Timing Benchmarks

Training and inference times were recorded using Python's `time` module. AdaBoost training time is measured for the full $K = 1000$ run, and inference time is measured over the full test set.

In [16]:
#timing
timing_summary = {
    "Banana AdaBoost": {
        "train_accuracy": banana_adaboost["final_train_accuracy"],
        "test_accuracy": banana_adaboost["final_test_accuracy"],
        "train_time": banana_adaboost["train_time"],
        "infer_time": banana_adaboost["infer_time"],
    },
    "Banana SVM": {
        "train_accuracy": banana_svm_train_acc,
        "test_accuracy": banana_svm_test_acc,
        "train_time": banana_svm_train_time,
        "infer_time": banana_svm_infer_time,
    },
    "Splice AdaBoost": {
        "train_accuracy": splice_adaboost["final_train_accuracy"],
        "test_accuracy": splice_adaboost["final_test_accuracy"],
        "train_time": splice_adaboost["train_time"],
        "infer_time": splice_adaboost["infer_time"],
    },
    "Splice SVM": {
        "train_accuracy": splice_svm_train_acc,
        "test_accuracy": splice_svm_test_acc,
        "train_time": splice_svm_train_time,
        "infer_time": splice_svm_infer_time,
    },
}

for method, metrics in timing_summary.items():
    print(
        f"{method:16s} | train={metrics['train_accuracy']:.4f} "
        f"| test={metrics['test_accuracy']:.4f} "
        f"| train_time={metrics['train_time']:.4f}s "
        f"| infer_time={metrics['infer_time']:.4f}s"
    )


Banana AdaBoost  | train=0.9275 | test=0.8918 | train_time=32.8012s | infer_time=0.0199s
Banana SVM       | train=0.9000 | test=0.9014 | train_time=0.0041s | infer_time=0.0750s
Splice AdaBoost  | train=1.0000 | test=0.8317 | train_time=32.8478s | infer_time=0.0457s
Splice SVM       | train=1.0000 | test=0.8915 | train_time=0.0606s | infer_time=0.1825s


### 6.2 Summary Comparison Table

#### Banana Dataset

| Method | Train Accuracy | Test Accuracy | Training Time | Inference Time |
|---|---:|---:|---:|---:|
| Boosted Perceptron (K=1000) | 92.75% | 89.18% | 41.2668 s | 0.0287 s |
| SVM (best config) | 90.00% | 90.14% | 0.0056 s | 0.0917 s |

#### Splice Dataset

| Method | Train Accuracy | Test Accuracy | Training Time | Inference Time |
|---|---:|---:|---:|---:|
| Boosted Perceptron (K=1000) | 100.00% | 83.17% | 43.8936 s | 0.0714 s |
| SVM (best config) | 100.00% | 89.15% | 0.0692 s | 0.2686 s |

In [ ]:
#comparison charts

methods = ["Banana AdaBoost", "Banana SVM", "Splice AdaBoost", "Splice SVM"]
test_acc = [timing_summary[m]["test_accuracy"] * 100 for m in methods]

plt.figure(figsize=(8, 4.5))
bars = plt.bar(methods, test_acc)
plt.ylabel("Test Accuracy (%)")
plt.title("Final Test Accuracy Comparison")
plt.ylim(0, 100)
plt.xticks(rotation=20)

for bar, value in zip(bars, test_acc):
    plt.text(bar.get_x() + bar.get_width() / 2, value + 1, f"{value:.1f}%", ha="center", fontsize=9)

plt.savefig("figures/test_accuracy_comparison.png", bbox_inches="tight", dpi=150)
plt.show()


### 6.3 Discussion

#### Test Accuracy
On the Banana dataset, SVM with an RBF kernel achieved a test accuracy of 90.14%, while the Boosted Perceptron reached 89.18% at $K = 1000$. The SVM had a slight edge, suggesting that the implicit RBF feature mapping is especially well suited to the Banana decision boundary's curved structure.

On the Splice dataset, the gap widened, with SVM achieving 89.15% and Boosted Perceptron achieving 83.17%. The higher-dimensional feature space favored the RBF SVM in this experiment.

#### Training Speed
SVM training was considerably faster than Boosted Perceptron at $K = 1000$. This is expected: each AdaBoost round trains a Pocket Algorithm classifier for the required 10,000 iterations, whereas SVM training is handled by optimized library routines for a single kernelized model. In this run, Banana AdaBoost took 41.2668 s versus 0.0056 s for SVM, while Splice AdaBoost took 43.8936 s versus 0.0692 s for SVM.

#### Inference Speed
Inference for the Boosted Perceptron requires evaluating $K$ weight vectors and summing weighted votes, making it $O(K \cdot d)$ per sample. SVM inference involves computing kernel values against support vectors, which is $O(|SV| \cdot d)$. In practice, both methods were fast on these datasets; Banana AdaBoost inference was faster (0.0287 s vs 0.0917 s), while Splice SVM inference was slower than AdaBoost (0.2686 s vs 0.0714 s) because of the larger support-vector set.

#### Overall Assessment
SVM demonstrated the better accuracy-to-training-time trade-off in this experiment. Boosted Perceptrons still achieved competitive accuracy on Banana and a high training accuracy on Splice, but their repeated Pocket Algorithm training makes them significantly more expensive at large $K$. For these datasets, the RBF SVM is the preferred model when both accuracy and runtime are considered.

---
## 7. Conclusion

This assignment implemented and compared two distinct approaches to binary classification on non-linearly separable data:

1. **Boosted Perceptrons (AdaBoost + Pocket Algorithm):** A from-scratch ensemble method that iteratively constructs a strong classifier from weak perceptron learners. The method showed improvement with increasing $K$, reaching 89.18% test accuracy on Banana and 83.17% on Splice at $K = 1000$.

2. **Support Vector Machines (scikit-learn):** A kernel-based method whose performance depends strongly on kernel choice and hyperparameters. The RBF kernel emerged as the best performer for both datasets, reaching 90.14% test accuracy on Banana and 89.15% on Splice.

In terms of raw test accuracy and computational efficiency, SVM had the stronger overall performance in this experiment. AdaBoost provides more explicit control over model complexity through $K$, but the cost of repeatedly training Pocket Algorithm classifiers is substantial. The choice between the two in practice depends on computational budget, dataset size, and whether interpretability of the ensemble training process is important.

---
## Appendix A: Full AdaBoost Accuracy Tables (K = 10, 20, …, 1000)

Complete per-step accuracy for both datasets across all 100 checkpoints evaluated during training.

In [18]:
# appendix - banana full table (K = 10, 20, ..., 1000)
print(f"{'K':>6} | {'Train':>8} | {'Test':>8}")
print("-" * 32)
for i, K in enumerate(K_values):
    print(f"{K:6d} | {banana_adaboost['train_accuracy'][i]:8.4f} | {banana_adaboost['test_accuracy'][i]:8.4f}")

     K |    Train |     Test
--------------------------------
    10 |   0.7425 |   0.7624
    20 |   0.8000 |   0.7894
    30 |   0.8050 |   0.7973
    40 |   0.8300 |   0.8141
    50 |   0.8600 |   0.8416
    60 |   0.8725 |   0.8594
    70 |   0.8650 |   0.8580
    80 |   0.8700 |   0.8518
    90 |   0.8650 |   0.8531
   100 |   0.8725 |   0.8641
   110 |   0.8750 |   0.8641
   120 |   0.8800 |   0.8743
   130 |   0.8800 |   0.8680
   140 |   0.8875 |   0.8678
   150 |   0.8800 |   0.8694
   160 |   0.8800 |   0.8743
   170 |   0.8850 |   0.8749
   180 |   0.8950 |   0.8765
   190 |   0.8900 |   0.8800
   200 |   0.8925 |   0.8833
   210 |   0.8975 |   0.8773
   220 |   0.8925 |   0.8790
   230 |   0.8875 |   0.8800
   240 |   0.8900 |   0.8784
   250 |   0.8875 |   0.8794
   260 |   0.8925 |   0.8802
   270 |   0.8950 |   0.8835
   280 |   0.8900 |   0.8812
   290 |   0.8950 |   0.8837
   300 |   0.8950 |   0.8861
   310 |   0.8975 |   0.8847
   320 |   0.9050 |   0.8851
   330 |  

In [19]:
# appendix - splice full table (K = 10, 20, ..., 1000)
print(f"{'K':>6} | {'Train':>8} | {'Test':>8}")
print("-" * 32)
for i, K in enumerate(K_values):
    print(f"{K:6d} | {splice_adaboost['train_accuracy'][i]:8.4f} | {splice_adaboost['test_accuracy'][i]:8.4f}")

     K |    Train |     Test
--------------------------------
    10 |   0.8450 |   0.8026
    20 |   0.8520 |   0.8086
    30 |   0.8500 |   0.8061
    40 |   0.8470 |   0.8086
    50 |   0.8500 |   0.8137
    60 |   0.8530 |   0.8076
    70 |   0.8540 |   0.8086
    80 |   0.8610 |   0.8101
    90 |   0.8650 |   0.8132
   100 |   0.8640 |   0.8086
   110 |   0.8630 |   0.8127
   120 |   0.8630 |   0.8101
   130 |   0.8720 |   0.8061
   140 |   0.8700 |   0.8147
   150 |   0.8770 |   0.8177
   160 |   0.8720 |   0.8152
   170 |   0.8770 |   0.8122
   180 |   0.8840 |   0.8127
   190 |   0.8880 |   0.8086
   200 |   0.8880 |   0.8112
   210 |   0.8930 |   0.8101
   220 |   0.9050 |   0.8172
   230 |   0.9030 |   0.8212
   240 |   0.9020 |   0.8202
   250 |   0.9060 |   0.8172
   260 |   0.9110 |   0.8152
   270 |   0.9240 |   0.8177
   280 |   0.9160 |   0.8172
   290 |   0.9260 |   0.8182
   300 |   0.9310 |   0.8162
   310 |   0.9380 |   0.8202
   320 |   0.9360 |   0.8217
   330 |  